<a href="https://colab.research.google.com/github/JulioRan09/Porgramacion-Predictiva/blob/main/Sesion12_Evaluacion_Datos_Categoricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: Julio Rangel García
- **Matrícula** 276572

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [1]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()
#df.shape

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
#df.info()

---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [3]:
# 1.1 — Recorre todas las columnas categóricas con .value_counts() o .unique()
# para inspeccionar sus valores.
col_categoricas = ['gender', 'SeniorCitizen', 'Partner', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                   'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


for col in col_categoricas:
  n_unicos = df[col].unique()
  print(f'{col} -- tiene los valores únicos: {n_unicos}')

gender -- tiene los valores únicos: ['Female' 'Male']
SeniorCitizen -- tiene los valores únicos: [0 1]
Partner -- tiene los valores únicos: ['Yes' 'No']
PhoneService -- tiene los valores únicos: ['No' 'Yes']
MultipleLines -- tiene los valores únicos: ['No phone service' 'No' 'Yes']
InternetService -- tiene los valores únicos: ['DSL' 'Fiber optic' 'No']
OnlineSecurity -- tiene los valores únicos: ['No' 'Yes' 'No internet service']
OnlineBackup -- tiene los valores únicos: ['Yes' 'No' 'No internet service']
DeviceProtection -- tiene los valores únicos: ['No' 'Yes' 'No internet service']
TechSupport -- tiene los valores únicos: ['No' 'Yes' 'No internet service']
StreamingTV -- tiene los valores únicos: ['No' 'Yes' 'No internet service']
StreamingMovies -- tiene los valores únicos: ['No' 'Yes' 'No internet service']
Contract -- tiene los valores únicos: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling -- tiene los valores únicos: ['Yes' 'No']
PaymentMethod -- tiene los valores úni

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_ Revisé todas las columnas categóricas y no encontré ninguna que parezca tener datos inconsistentes. El más cercano son las columnas como "OnlineSecurity" que pareciera tener duplicados los datos "No" y "No internet Service" pero pueden siginificar que sí tiene internet pero no tiene Seguridad.

---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [4]:
df['OnlineSecurity'].value_counts()

,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [5]:
df['InternetService'].value_counts()


,count
InternetService,
Fiber optic,3096
DSL,2421
No,1526


En la celda anterior, podemos tener un adelanto antes de ejecutar pd.crosstab. Observamos que con value.counts(), la cantidad de datos con "No internet service" e Internet Service = "no" son exactamente 1526 para ambos casos, podemos esperar que todas las filas coincidan.

In [6]:
# 2.1 — Verifica: ¿las filas con 'No internet service' en OnlineSecurity
# coinciden con las filas donde InternetService == 'No'?
# (pista: cruza ambas columnas con pd.crosstab o filtrando)
pd.crosstab(df['OnlineSecurity'], df['InternetService'])


InternetService,DSL,Fiber optic,No
OnlineSecurity,,,
No,1241,2257,0
No internet service,0,0,1526
Yes,1180,839,0


Con pd.crosstab confirmamos que efectivamente, las 1526 filas donde Internet services es "No" coinciden con OnlineSecurity "No internet service"

**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_ Es una categoría legítima, es para los casos donde el usuario no tiene servicio de internet y por ende, no tiene OnlineSecurity. Estos usuarios son diferentes de aquellos que no tienen OnlineSecurity pero sí tienen internet.

---
## Actividad 3 — Alta cardinalidad (25 pts)

In [7]:
# 3.1 — Calcula .nunique() para TODAS las columnas del dataset (no solo las categóricas)
# y la razón (valores únicos / total de filas) para cada una.

for col in df:
    n_unicos = df[col].nunique()
    ratio = n_unicos / len(df)
    print(f'{col:18s} -> {n_unicos:5d} valores únicos  ({ratio:.2%} del total de filas)')


customerID         ->  7043 valores únicos  (100.00% del total de filas)
gender             ->     2 valores únicos  (0.03% del total de filas)
SeniorCitizen      ->     2 valores únicos  (0.03% del total de filas)
Partner            ->     2 valores únicos  (0.03% del total de filas)
Dependents         ->     2 valores únicos  (0.03% del total de filas)
tenure             ->    73 valores únicos  (1.04% del total de filas)
PhoneService       ->     2 valores únicos  (0.03% del total de filas)
MultipleLines      ->     3 valores únicos  (0.04% del total de filas)
InternetService    ->     3 valores únicos  (0.04% del total de filas)
OnlineSecurity     ->     3 valores únicos  (0.04% del total de filas)
OnlineBackup       ->     3 valores únicos  (0.04% del total de filas)
DeviceProtection   ->     3 valores únicos  (0.04% del total de filas)
TechSupport        ->     3 valores únicos  (0.04% del total de filas)
StreamingTV        ->     3 valores únicos  (0.04% del total de filas)
Stre

**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_ Las columnas con mayor cardinalidad son "CustomerID" con 100%, "Total Charges" con 92.73% y "MonthlyCharges" con 22.50%. Estas columnas no suelen se útiles para utilizarse como variables predictoras porque no generan patrones, cada valor pertenece a un individuo.

**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [8]:
df['TotalCharges'].value_counts().head(10)

,count
TotalCharges,
,11
20.2,11
19.75,9
20.05,8
19.9,8
19.65,8
19.55,7
45.3,7
20.15,6


En esta sección del código, descubrimos que hay un grupo de 11 elementos vacíos en la columna TotalCharges, después de analizarlo, las medidas de tendencia no parecen ser buenas opciones para reemplazar estos valores, la moda cargará demasiado un valor de totalcharge, la media puede estar sesgada. Por este análisis, hemos tomado la decisión de descartar los 11 valores en el código a continuación:

In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['TotalCharges'].notna()]
df['TotalCharges'].value_counts().head(10)

,count
TotalCharges,
20.20,11
19.75,9
19.65,8
19.90,8
20.05,8
45.30,7
19.55,7
19.45,6
20.25,6


Adelantando a pasos siguientes, agrupar todos los valores fuera de un umbral, o de un head(10,20,30) no tiene mucho sentido, en su lugar crearemos una nueva columna que divida la columna TotalCharges en deciles, y cada décil se le asignará una variable numérica que representa su posición. De esta manera reducimos la cardinalidad de 92.86% a 0.14%.

In [10]:
df['TotalChargesGroup'] = pd.qcut(df['TotalCharges'],
                                    q=10,  # deciles
                                    labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

for col in df:
    n_unicos = df[col].nunique()
    ratio = n_unicos / len(df)
    print(f'{col:18s} -> {n_unicos:5d} valores únicos  ({ratio:.2%} del total de filas)')


customerID         ->  7032 valores únicos  (100.00% del total de filas)
gender             ->     2 valores únicos  (0.03% del total de filas)
SeniorCitizen      ->     2 valores únicos  (0.03% del total de filas)
Partner            ->     2 valores únicos  (0.03% del total de filas)
Dependents         ->     2 valores únicos  (0.03% del total de filas)
tenure             ->    72 valores únicos  (1.02% del total de filas)
PhoneService       ->     2 valores únicos  (0.03% del total de filas)
MultipleLines      ->     3 valores únicos  (0.04% del total de filas)
InternetService    ->     3 valores únicos  (0.04% del total de filas)
OnlineSecurity     ->     3 valores únicos  (0.04% del total de filas)
OnlineBackup       ->     3 valores únicos  (0.04% del total de filas)
DeviceProtection   ->     3 valores únicos  (0.04% del total de filas)
TechSupport        ->     3 valores únicos  (0.04% del total de filas)
StreamingTV        ->     3 valores únicos  (0.04% del total de filas)
Stre

**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_ Esta técnica funciona bien cuando la densidad de valores/categorías está cargado en unas pocas variables. Para el caso de Country, unos pocos países tenían la mayoría de registros. En el caso de este notebook, los 6530 valores apenas formaban grupos de 11 por cada valor.

---
## Actividad 4 — Tipos de dato (30 pts)

In [11]:
df['TotalCharges'].dtype

dtype('float64')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_ Tengo la teoría de que lo pudo cargar como object debido a los valores vacíos que quitamos en las celdas anteriores. Probaremos a ver si este error ha desaparecido.

In [12]:
# 4.2 — Corrige el tipo de TotalCharges.
# Pista: pd.to_numeric() con el parámetro errors= te puede ayudar a identificar
# o manejar los valores problemáticos que encontraste en 4.1.


Cuando quitamos los valores vacíos de la columna TotalCharges y lo convertimos a numérico se ha corregido el problema y ahora pandas lo detecta como tipo flotante. Revisemos por curiosidad la columna MonthlyCharges:

In [13]:
df['MonthlyCharges'].dtype
#Esta está bien, no hay que corregirla.

dtype('float64')

In [14]:
# 4.3 — Convierte a category las columnas categóricas que, según lo que calculaste
# en la Actividad 3, tengan cardinalidad baja y valores fijos.
# Verifica con .dtypes que el cambio se aplicó correctamente.

for col in col_categoricas:
  df[col] = df[col].astype('category')

df.dtypes


,0
customerID,object
gender,category
SeniorCitizen,category
Partner,category
Dependents,object
tenure,int64
PhoneService,category
MultipleLines,category
InternetService,category
OnlineSecurity,category


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?

2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_
1. Los problemas de formato, valores inválidos son más fáciles de identificar y tomar decisiones sobre como manejarlos. En particular, este dataset estaba bastante limpio y no fue complicado manejar este rubro.
El tema de la cardinalidad se comportó como se esperaba, los valores con alta cardinalidad eran las columnas esperadas de acuerdo a su naturaleza.

2. Para el caso específico del CustomerID, no tengo mucho que decir, es un identificador único y no debería ser tomado en cuenta para modelos predictivos. En el caso de TotalCharges y MonthlyCharges creo que se le puede sacar provecho reduciendo su cardinalidad como lo hicimos en este notebook, habría que hacer doublecheck para verificar que lo que hicimos tiene sentido revisando la frecuencia de datos de cada decil, todos quedaron con aproximadamente la cantidad de datos, por lo que parece que sí se trataba de una distribución uniforme.

---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.